## 1 - Data Analysis

El objetivo principal es extraer y analizar los datos de la base de datos ***MIMIC-IV***
El primer paso es analizar de que tipo de datos disponemos, de primeras se puede observar que el formato de los archivos es `csv.gz`, el formato `gz` indica que es un archivo comprimido, y previamente tiene el formato `csv`, por lo tanto se puede llegar a suponer que es un archivo csv comprimido. La libreria pandas ayudara a leer los archivos csv comprimidos directamente sin descompression manual.

Lo prinicipal es importar las librerias que seran necesarias para analizar y procesar estos datos

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

Con las librerias requeridas cargadas ya se pueden importar y empezar a usar los datos, aunque antes hay que entender como estan estructurados y que relacion hay entre diferentes archivos `csv`.

Segun la documentacion de **MIMIC-IV** tenemos dos directorios principales:

- **hosp**: Contiene **546,028** registros unicos de hospitalizaciones para **223,452** individiuos unicos. Estos datos se dividen en cuatro categorias principales:
    - **Información Básica y Logística**: Incluye los datos personales de los pacientes (edad, genero, etc), y registra cuando entra y sale un paciente de una unidad en concreto.
    - **Datos Clínicos y de Laboratorio**: Contiene eventos registrados en un laboratorio, como por ejemplo resultados de analisis de sangre, orina, etc.
    - **Medicamentos**: Contiene las recetas prescritas por los medicos, tambien incluye un registro de cuando y como se ha administrado el medicamiento.
    - **Facturación y Diagnósticos**: Inluye los datos que indican los codigos de los diagnosticos y procedimientos realizados.

- **icu**: Contiene **94,458** registros de estancias de pacientes en la UCI. Estos datos se estructuran de forma que hay una tabla "estancias" central que se conecta con otras tablas de eventos, estas tablas registran todos los sucesos de un paciente minuto a minuto.

Con este conocimiento solo queda conocer los campos que enlazan las diferentes tablas, los campos mas importantes son:

- `subject_id`: Identificador unico del paciente
- `hadm_id`: Identificador unico de admision hospitalaria (un paciente puede tener mas de una admision)
- `stay_id`: Identificador unico de estancia en la UCI (una admission puede tener varias estancias)

In [2]:
ID_SUBJECT = 'subject_id'
ID_ADMISION = 'hadm_id'
ID_STAY = 'stay_id'

El primer paso se tratara de analizar los diagnosticos del dataset usando la variable `icd_code`, esta variable nos indica el codigo ICD del diagnostico, este codigo usa un sistema estandarizado para codificar enfermedades, signos o sintomas, actualmente existen dos principales revisiones: **ICD-9** y **ICD-10**. Estas dos revisiones varian sobretodo en la cantidad de codigos y estructura, el objetivo es usar estos codigos para identificar en el dataset los diagnosis que contengan casos de sepsis, o algun sintoma o enfermedad que nos ayude a indicar la sepsis, se usara la variable `icd_version` para identificar la version en los datos.


### ICD-9

| Código(s) | Descripción |
|-----------|-------------|
| **995.91** | Sepsis (SIRS de origen infeccioso) |
| **995.92** | Sepsis grave |
| **785.52** | Choque séptico |
| **038.0–038.9** | Septicemia (038.11 Staph aureus, 038.42 E. coli, 038.9 no especificada) |
| **771.81** | Sepsis neonatal |

### ICD-10

| Código(s) | Descripción |
|-----------|-------------|
| **A41.9** | Sepsis, organismo no especificado (el más común en registros) |
| **A40.0, A40.1, A40.3, A40.8, A40.9** | Sepsis por diferentes grupos de Estreptococos |
| **A41.01** | Sepsis por *Staphylococcus aureus* sensible a meticilina (MSSA) |
| **A41.02** | Sepsis por *Staphylococcus aureus* resistente a meticilina (MRSA) |
| **A41.51** | Sepsis por *Escherichia coli* |
| **A41.52** | Sepsis por *Pseudomonas* |
| **A41.81** | Sepsis por *Enterococcus* |
| **A41.89** | Sepsis por otros organismos especificados (incluye sepsis viral) |
| **A02.1** | Sepsis por *Salmonella* |
| **B37.7** | Sepsis candidiásica (fúngica) |
| **R65.20** | Sepsis grave sin choque séptico |
| **R65.21** | Sepsis grave con choque séptico |
| **P36.0–P36.9** | Sepsis bacteriana neonatal |
| **O85** | Sepsis puerperal (postparto) |
| **O75.3** | Sepsis durante el trabajo de parto |
| **T81.44** | Sepsis tras procedimiento quirúrgico |

In [3]:
diagnoses = pd.read_csv('../data/hosp/diagnoses_icd.csv.gz')
print(diagnoses.shape)
print(diagnoses.dtypes)
print(diagnoses.head())

(6364488, 5)
subject_id     int64
hadm_id        int64
seq_num        int64
icd_code         str
icd_version    int64
dtype: object
   subject_id   hadm_id  seq_num icd_code  icd_version
0    10000032  22595853        1     5723            9
1    10000032  22595853        2    78959            9
2    10000032  22595853        3     5715            9
3    10000032  22595853        4    07070            9
4    10000032  22595853        5      496            9


In [4]:
ICD_CODE = 'icd_code' # Identificador del codigo de ICD 
ICD_VERSION = 'icd_version' # Identificador de la version de ICD

# ---------------------------------- ICD 9 ----------------------------------

# Definimos los codigos especificos de ICD9
SEPSIS_ICD9_EXPLICIT = {
    '99591',  # Sepsis
    '99592',  # Sepsis grave
    '78552',  # Choque séptico
    '77181',  # Sepsis neonatal
}

# Filtramos por version
icd9_df = diagnoses[diagnoses[ICD_VERSION] == 9].copy()

# Incluimos los diagnosis con codigo que empieza por 038 (Septicemia)
icd9_mask = (
    icd9_df[ICD_CODE].isin(SEPSIS_ICD9_EXPLICIT) |
    icd9_df[ICD_CODE].str.startswith('038')
)

# Evitamos registros duplicados identificandolos por admision
sepsis_icd9_hadm = icd9_df[icd9_mask][ID_ADMISION].unique()

# ---------------------------------- ICD 10 ----------------------------------

# Prefijos A40.x y A41.x, se capturan todos los subtipos
ICD10_SEPSIS_PREFIXES = ('A40', 'A41')

# Códigos adicionales fuera de A40/A41
ICD10_SEPSIS_EXTRA = {
    'A021',   # Salmonella sepsis
    'B377',   # Candida sepsis (fúngica)
    'R6520',  # Sepsis grave sin choque séptico
    'R6521',  # Sepsis grave con choque séptico
    'O85',    # Sepsis puerperal
    'O753',   # Sepsis durante el trabajo de parto
    'T8144',  # Sepsis postquirúrgica
}

# Prefijos de sepsis neonatal P36.x
ICD10_NEONATAL_PREFIX = ('P36',)

# Filtramos por version y incluimos todos los codigos
icd10_mask = (
    (diagnoses[ICD_VERSION] == 10) &
    (
        diagnoses[ICD_CODE].str.startswith(ICD10_SEPSIS_PREFIXES) |
        diagnoses[ICD_CODE].str.startswith(ICD10_NEONATAL_PREFIX) |
        diagnoses[ICD_CODE].isin(ICD10_SEPSIS_EXTRA)
    )
)

# Evitamos registros duplicados identificandolos por admision
sepsis_icd10_hadm = diagnoses[icd10_mask][ID_ADMISION].unique()

# Unimos todos los registros que contienen ICD9 y ICD10
sepsis_hadm_ids = np.union1d(sepsis_icd10_hadm, sepsis_icd9_hadm)

print(f'Admisiones con sepsis (ICD-9):  {len(sepsis_icd9_hadm):,}')
print(f'Admisiones con sepsis (ICD-10): {len(sepsis_icd10_hadm):,}')
print(f'Total admisiones:        {len(sepsis_hadm_ids):,}')

Admisiones con sepsis (ICD-9):  8,705
Admisiones con sepsis (ICD-10): 13,763
Total admisiones:        22,467


Despues de indicar los codigos y versiones ICD se puede observar que hay un total de **22,467** resultados de admisiones, teniendo en cuenta el total de admisiones se puede inferir que solo indicar los codigos ICD no es suficiente, esto se debe a que se estan capturando solo los casos donde la sepsis esta codificada de forma explicita, pero puede suceder que haya muchos casos que no esten codificados explicitamente y tengan otros indicadores que nos puede ayudar a llegar a la conclusion de que se trata de una sepsis. Para acabar de completar los datos es posible seguir dos estrategias que ya son fiables y un estandar en el sector medico:

- **Algoritmo Angus**: Muchos casos de sepsis en ICD-9 eran registrados como **Infección + Disfunción orgánica**, al usar estos indicadores juntos es posible obtener mas resultados de diagnosis de sepsis.

- **Sepsis-3**: Requiere usar el indicador **SOFA** junto a si el diagnosis entra dentro de la categoria de infeccion, al usar este calculo es posible filtrar casos de sepsis, implementarlo implica anadir mucha complejidad al analisis.

Para el primer analisis se usara solo el algoritmo angus para no anadir demasiada complejidad.

In [5]:
# ---------------------------------- Algoritmo Angus ----------------------------------

#
ANGUS_SINGLE_CODES = {'590', '597', '5990', '6816', '9966', '9985', '9993'}

# rangos de infección
ANGUS_INFECTION_RANGES = [
    (1,   41),   # Infecciones bacterianas y sistémicas
    (90,  104),  # Sífilis y espiroquetas
    (110, 118),  # Micosis
    (320, 325),  # Infecciones SNC
    (461, 486),  # Infecciones respiratorias / neumonía
    (540, 542),  # Apendicitis / abdomen
    (566, 567),  # Peritonitis
]

def is_icd9_angus(code: str) -> bool:
    """Devuelve True si el codigo entra dentro del rango de infeccion definido por el algoritmo Angus"""
    try:
        numeric = int(code[:3])
        return any(lo <= numeric <= hi for lo, hi in ANGUS_INFECTION_RANGES)
    except:
        return False


icd9_angus_df = diagnoses[diagnoses[ICD_VERSION] == 9].copy()
    
icd9_angus_mask = (
    icd9_angus_df[ICD_CODE].isin(ANGUS_SINGLE_CODES) |
    icd9_angus_df[ICD_CODE].apply(is_icd9_angus)
)

icd9_angus_df = icd9_angus_df[icd9_angus_mask][ID_ADMISION].unique()
sepsis_hadm_ids = np.union1d(sepsis_icd10_hadm, icd9_angus_df)

print(f'Admisiones con sepsis con algoritmo Angus (ICD-9): {len(icd9_angus_df):,}')
print(f"Total admisiones:        {len(sepsis_hadm_ids):,}")

Admisiones con sepsis con algoritmo Angus (ICD-9): 63,066
Total admisiones:        76,828


Al aplicar el algoritmo Angus se consigue un total de **76,828** admisiones. Ahora que tenemos los diagnosticos de las admisiones nos falta determinar el momento exacto en el que el paciente desarrolla la patologia. Es necesario observar cuando hay por ejemplo un incremento de dos puntos en el score SOFA, para conseguir este objetivo es necesario cruzar la tabla de diagnosis con las tablas `transfers` y `icustays`. Las tablas establecen una linea de tiempo que nos permite observar los eventos previos a parte de cuando ocurre un incremento en los valores criticos.

Aunque ahora hay una definicion de las dos tablas necesarias para establecer una linea de tiempo es necesario anadir otra tabla que ayuda a identificar la admision y terminar de crear la conexion entre las diferentes tablas, la tabla que se usara para este proposito es la tabla `admissions`.

In [6]:
admissions = pd.read_csv('../data/hosp/admissions.csv.gz')
transfers = pd.read_csv('../data/hosp/transfers.csv.gz')
icustays = pd.read_csv('../data/icu/icustays.csv.gz')

print('-------------------------------- Tabla admissions --------------------------------')
print(admissions.shape)
print(admissions.dtypes)

print('-------------------------------- Tabla icustays --------------------------------')
print(icustays.shape)
print(icustays.dtypes)

print('-------------------------------- Tabla transfers --------------------------------')
print(transfers.shape)
print(transfers.dtypes)

-------------------------------- Tabla admissions --------------------------------
(546028, 16)
subject_id              int64
hadm_id                 int64
admittime                 str
dischtime                 str
deathtime                 str
admission_type            str
admit_provider_id         str
admission_location        str
discharge_location        str
insurance                 str
language                  str
marital_status            str
race                      str
edregtime                 str
edouttime                 str
hospital_expire_flag    int64
dtype: object
-------------------------------- Tabla icustays --------------------------------
(94458, 8)
subject_id          int64
hadm_id             int64
stay_id             int64
first_careunit        str
last_careunit         str
intime                str
outtime               str
los               float64
dtype: object
-------------------------------- Tabla transfers --------------------------------
(2413581, 7)
s

Despues de cargar las dos tablas se puede observar que las dos contienen la variable `hadm_id` que nos permite identificar la admision, aparte de las otras variables que ayudan a identificar la estancia y el paciente (`stay_id` y `subject_id`). El siguiente paso es enlazar las diferentes tablas que ya se han cargado.

In [7]:
sepsis_df = pd.DataFrame({ID_ADMISION: list(sepsis_hadm_ids)})

cohort_icu = sepsis_df.merge(
    admissions[['subject_id', ID_ADMISION, 'admittime']],
    on=ID_ADMISION,
    how='inner'
)

final_cohort = cohort_icu.merge(
    icustays[['subject_id', ID_ADMISION, ID_STAY, 'intime', 'outtime']], 
    on=['subject_id', ID_ADMISION],
    how='inner'
)

print(final_cohort.shape)
print(final_cohort.dtypes)
print(final_cohort.head())

(29966, 6)
hadm_id       int64
subject_id    int64
admittime       str
stay_id       int64
intime          str
outtime         str
dtype: object
    hadm_id  subject_id            admittime   stay_id               intime  \
0  20002950    18596567  2134-06-21 20:57:00  32853022  2134-06-21 21:52:00   
1  20003014    15272409  2146-05-07 17:50:00  34079238  2146-05-07 23:01:50   
2  20003427    13927771  2184-05-14 20:15:00  35250851  2184-05-23 20:20:50   
3  20003427    13927771  2184-05-14 20:15:00  35475938  2184-05-14 21:45:00   
4  20003491    11540283  2197-12-18 04:50:00  33494445  2197-12-18 06:10:00   

               outtime  
0  2134-06-22 16:25:45  
1  2146-05-09 03:32:21  
2  2184-05-24 02:10:52  
3  2184-05-23 20:20:28  
4  2197-12-20 19:02:58  


Una vez unidas las tablas de estancias y transferencias se puede observar que obtenemos un total de **29,966 registros**, esta tabla contiene la fecha y hora de entrada y salida del paciente por cada admision, incluyendo tambien la estancia. El siguiente paso es extraer las variables temporales para construir la **memoria** del modelo. El objetivo es priorizar la extraccion de variables que permiten identificar mejor la patologia, por ejemplo el score **SOFA**, este es el estandar clinico de referencia que se puede usar. Existen dos tablas que se pueden usar para este proposito:

- **chartevents**: Contiene valores de los signos vitales como la frecuencia cardiaca, presion arterial media, saturacion de oxigeno y temperatura.

- **labevents**: Incluye valores como el lactato, creatinina, bilirrubina, recuento de plaquetas y relacion PaO2/FiO2.

En resumen la tabla `chartevents` contiene valores vitales y `labevents` en cambio contiene valores a partir de analisis de laboratorio.

In [8]:
chartevents = pd.read_csv('../data/icu/chartevents.csv.gz')
labevents = pd.read_csv('../data/hosp/labevents.csv.gz')

print('-------------------------------- Tabla chartevents --------------------------------')
print(chartevents.shape)
print(chartevents.dtypes)
print(chartevents.head())

print('-------------------------------- Tabla labevents --------------------------------')
print(labevents.shape)
print(labevents.dtypes)
print(labevents.head())

-------------------------------- Tabla chartevents --------------------------------
(432997491, 11)
subject_id        int64
hadm_id           int64
stay_id           int64
caregiver_id    float64
charttime           str
storetime           str
itemid            int64
value               str
valuenum        float64
valueuom            str
warning         float64
dtype: object
   subject_id   hadm_id   stay_id  caregiver_id            charttime  \
0    10000032  29079034  39553978       18704.0  2180-07-23 12:36:00   
1    10000032  29079034  39553978       18704.0  2180-07-23 12:36:00   
2    10000032  29079034  39553978       18704.0  2180-07-23 12:36:00   
3    10000032  29079034  39553978       18704.0  2180-07-23 14:00:00   
4    10000032  29079034  39553978       18704.0  2180-07-23 14:00:00   

             storetime  itemid              value  valuenum valueuom  warning  
0  2180-07-23 14:45:00  226512               39.4      39.4       kg      0.0  
1  2180-07-23 14:45:00  22670

Ahora que las tablas requeridas estan cargadas es posible empezar a seleccionar las variables que usaremos para crear el modelo, para empezar se seleccionaran las variables que cumplan con el criterio del score SOFA. MIMIC-IV usa el campo `itemid` para identificar por codigo cada variable posible, por lo tanto hay que cruzar estas dos tablas con la tabla `d_items` que incluye los registros de las diferentes variables como constantes, y la tabla `d_labitems` para variables en el laboratorio.

In [9]:
d_items = pd.read_csv('../data/icu/d_items.csv.gz')
d_labitems = pd.read_csv('../data/hosp/d_labitems.csv.gz')

print('-------------------------------- Tabla d_items --------------------------------')
print(d_items.shape)
print(d_items.dtypes)
print(d_items.head())

print('-------------------------------- Tabla d_labitems --------------------------------')
print(d_labitems.shape)
print(d_labitems.dtypes)
print(d_labitems.head())

-------------------------------- Tabla d_items --------------------------------
(4095, 9)
itemid               int64
label                  str
abbreviation           str
linksto                str
category               str
unitname               str
param_type             str
lownormalvalue     float64
highnormalvalue    float64
dtype: object
   itemid                    label        abbreviation         linksto  \
0  220001             Problem List        Problem List     chartevents   
1  220003       ICU Admission date  ICU Admission date  datetimeevents   
2  220045               Heart Rate                  HR     chartevents   
3  220046  Heart rate Alarm - High     HR Alarm - High     chartevents   
4  220047   Heart Rate Alarm - Low      HR Alarm - Low     chartevents   

              category unitname     param_type  lownormalvalue  \
0              General      NaN           Text             NaN   
1                  ADT      NaN  Date and time             NaN   
2  Routine

Ya estan cargadas todas las tablas necesarias, el paso final es terminar de cruzar las tablas y extraer los datos de las 24 a 48 horas previsas al inicio de un episodio de sepsis, separando los eventos que han sucedido previamente en intervalos de 1 hora.

Para empezar el proceso de extraccion de los datos mencionados el primer paso es usar la tabla `final_cohort` y usar las fechas de entrada y salida para tenerlos eventos con un rango establecido.

In [10]:
# Convertir campos de texto a campos de fecha y hora de forma explicita
final_cohort['intime'] = pd.to_datetime(final_cohort['intime'])
final_cohort['outtime'] = pd.to_datetime(final_cohort['outtime'])

# Punto de referencia de entrada a UCI
final_cohort['sepsis_onset'] = final_cohort['intime']

# Ventana de extracción
HOURS_BEFORE = 48  # o 24
final_cohort['window_start'] = final_cohort['sepsis_onset'] - pd.Timedelta(hours=HOURS_BEFORE)
final_cohort['window_end']   = final_cohort['sepsis_onset']

El siguiente paso es filtrar las tablas `d_items` y `d_labitems` para obtener solo las variables que son utiles para el objetivo.

In [11]:
# SOFA - signos vitales (chartevents via d_items)
VITAL_ITEMIDS = {
    220045,  # Heart Rate
    220179,  # Non Invasive Blood Pressure systolic
    220180,  # Non Invasive Blood Pressure diastolic
    220052,  # Arterial Blood Pressure mean
    220210,  # Respiratory Rate
    223762,  # Temperature Celsius
    220277,  # SpO2 / Pulse Oximetry
    229867,  # FiO2 (ventilador)
    220339,  # GCS - Eye Opening
    223900,  # GCS - Verbal Response
    223901,  # GCS - Motor Response
}

# SOFA - laboratorio (labevents via d_labitems)
LAB_ITEMIDS = {
    51265,   # Platelet Count
    50912,   # Creatinine
    50885,   # Bilirubin Total
    50821,   # PaO2 (Blood Gas)
    50820,   # pH
    50813,   # Lactate
    51006,   # Blood Urea Nitrogen
}

print(d_items[d_items['itemid'].isin(VITAL_ITEMIDS)][['itemid','label','unitname']])
print(d_labitems[d_labitems['itemid'].isin(LAB_ITEMIDS)][['itemid','label','fluid']])

     itemid                                  label  unitname
2    220045                             Heart Rate       bpm
8    220052           Arterial Blood Pressure mean      mmHg
24   220179   Non Invasive Blood Pressure systolic      mmHg
25   220180  Non Invasive Blood Pressure diastolic      mmHg
28   220210                       Respiratory Rate  insp/min
36   220277            O2 saturation pulseoxymetry         %
40   220339                               PEEP set     cmH2O
338  223762                    Temperature Celsius        °C
407  223900                  GCS - Verbal Response       NaN
408  223901                   GCS - Motor Response       NaN
     itemid             label  fluid
11    50813           Lactate  Blood
18    50820                pH  Blood
19    50821               pO2  Blood
83    50885  Bilirubin, Total  Blood
110   50912        Creatinine  Blood
202   51006     Urea Nitrogen  Blood
450   51265    Platelet Count  Blood


El paso a seguir a continuacion es cruzar la tabla `chartevents` con la tabla `d_items`.

In [12]:
# IDs de estancias en la cohorte
cohort_stay_ids = set(final_cohort['stay_id'].unique())

# Filtrar chartevents: solo cohorte + solo itemids relevantes
chart_filtered = chartevents[
    chartevents['stay_id'].isin(cohort_stay_ids) &
    chartevents['itemid'].isin(VITAL_ITEMIDS) &
    chartevents['valuenum'].notna()
][['subject_id', 'hadm_id', 'stay_id', 'itemid', 'charttime', 'valuenum']].copy()

chart_filtered['charttime'] = pd.to_datetime(chart_filtered['charttime'])

# Unir con d_items para tener el label
chart_filtered = chart_filtered.merge(
    d_items[['itemid', 'label', 'unitname']],
    on='itemid',
    how='left'
)

# Aplicar ventana temporal usando stay_id como llave
chart_windowed = chart_filtered.merge(
    final_cohort[['stay_id', 'window_start', 'window_end']],
    on='stay_id',
    how='inner'
)

chart_windowed = chart_windowed[
    (chart_windowed['charttime'] >= chart_windowed['window_start']) &
    (chart_windowed['charttime'] <  chart_windowed['window_end'])
]

print(f'Registros chartevents en ventana: {len(chart_windowed):,}')

Registros chartevents en ventana: 31,600


Finalmente, cruzamos las tablas `labevents` y `d_labitems`

In [13]:
# Filtrar labevents: solo pacientes de la cohorte + itemids relevantes
cohort_subject_ids = set(final_cohort['subject_id'].unique())

lab_filtered = labevents[
    labevents['subject_id'].isin(cohort_subject_ids) &
    labevents['itemid'].isin(LAB_ITEMIDS) &
    labevents['valuenum'].notna()
][['subject_id', 'hadm_id', 'itemid', 'charttime', 'valuenum']].copy()

lab_filtered['charttime'] = pd.to_datetime(lab_filtered['charttime'])

# Unir con d_labitems para tener el label
lab_filtered = lab_filtered.merge(
    d_labitems[['itemid', 'label', 'fluid']],
    on='itemid',
    how='left'
)

# Fix: convertir hadm_id a Int64 nullable para merge correcto (labevents tiene NaN)
# El merge solo por subject_id cruzaba labs de admisiones distintas del mismo paciente
lab_filtered['hadm_id'] = lab_filtered['hadm_id'].astype('Int64')
final_cohort['hadm_id'] = final_cohort['hadm_id'].astype('Int64')

# Asignar stay_id: merge por subject_id + hadm_id para anclar cada lab
# a la admisión correcta (evita cruzar labs de otras admisiones del mismo paciente)
lab_with_stay = lab_filtered.merge(
    final_cohort[['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime', 'window_start', 'window_end']],
    on=['subject_id', 'hadm_id'],
    how='inner'
)

# El lab corresponde a esta estancia si charttime está dentro de [intime, outtime]
lab_with_stay = lab_with_stay[
    (lab_with_stay['charttime'] >= lab_with_stay['intime']) &
    (lab_with_stay['charttime'] <  lab_with_stay['outtime'])
]

# Aplicar ventana de 48h
lab_windowed = lab_with_stay[
    (lab_with_stay['charttime'] >= lab_with_stay['window_start']) &
    (lab_with_stay['charttime'] <  lab_with_stay['window_end'])
]

print(f'Tras merge con hadm_id: {len(lab_with_stay):,}')
print(f'Registros labevents en ventana 48h: {len(lab_windowed):,}')

Tras merge con hadm_id: 1,591,991
Registros labevents en ventana 48h: 0


Una vez tenemos las diferentes tablas cruzadas con todos los registros cumpliendo una ventana de tiempo especifica ya es posible separar estos datos en los intervalos que se definan, en este caso se definira un intervalo de 1 hora entre eventos.

In [14]:
def add_time_features(df, onset_col='window_end'):
    """
    Añade columna 'hours_before_onset' (negativo = antes del onset)
    y 'hour_bucket' para agrupar en intervalos de 1h.
    """
    df = df.merge(
        final_cohort[['stay_id', 'sepsis_onset']],
        on='stay_id',
        how='left'
    )
    # Tiempo relativo al onset en horas (negativo = previo al onset)
    df['hours_before_onset'] = (
        (df['charttime'] - df['sepsis_onset'])
        .dt.total_seconds() / 3600
    )
    # Bucket: floor al entero = hora entera (ej: -23.5h → bucket -24)
    df['hour_bucket'] = df['hours_before_onset'].apply(
        lambda h: int(h) if h == int(h) else int(h) - (1 if h < 0 else 0)
    )
    # Alternativa más limpia con floor:
    # df['hour_bucket'] = np.floor(df['hours_before_onset']).astype(int)
    return df

chart_windowed = add_time_features(chart_windowed)
lab_windowed   = add_time_features(lab_windowed)

# Verificar distribución de buckets
print(chart_windowed['hour_bucket'].value_counts().sort_index().head(5))

hour_bucket
-48    53
-47    69
-46    55
-45    63
-44    67
Name: count, dtype: int64


Ahora el ultimo paso es convertir esta extraccion en una matriz para poder construir el modelo LSTM, la matriz va a contener los valores de estancia, el intervalo del evento y las diferentes variables que se han elegido para el modelo con el valor que tenia el paciente en ese momento. Un punto muy fuerte a tener en cuenta es que no usamos directamente el paciente si no que usamos la estancia, de esta forma evitamos anadir sesgos que puedan prevenir de alguna variable del paciente.

In [15]:
# Agregar: media por bucket (hay múltiples mediciones por hora)
chart_agg = (
    chart_windowed
    .groupby(['stay_id', 'hour_bucket', 'label'])['valuenum']
    .mean()
    .reset_index()
    .pivot_table(index=['stay_id', 'hour_bucket'], columns='label', values='valuenum')
    .reset_index()
)

lab_agg = (
    lab_windowed
    .groupby(['stay_id', 'hour_bucket', 'label'])['valuenum']
    .mean()
    .reset_index()
    .pivot_table(index=['stay_id', 'hour_bucket'], columns='label', values='valuenum')
    .reset_index()
)

# Unir vitales + laboratorio en un único DataFrame
features_df = chart_agg.merge(lab_agg, on=['stay_id', 'hour_bucket'], how='outer')
features_df = features_df.sort_values(['stay_id', 'hour_bucket'])

print(features_df.shape)
print(features_df.columns.tolist())

(8059, 12)
['stay_id', 'hour_bucket', 'Arterial Blood Pressure mean', 'GCS - Motor Response', 'GCS - Verbal Response', 'Heart Rate', 'Non Invasive Blood Pressure diastolic', 'Non Invasive Blood Pressure systolic', 'O2 saturation pulseoxymetry', 'PEEP set', 'Respiratory Rate', 'Temperature Celsius']
